In [52]:
# ============================================================
# MODELO CHURN MEJORADO - SIN LEAKAGE
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# ============================================================
# VARIABLES
# ============================================================

variables = [
    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas",
    "descuento_activo",
    "importe_total",
    "dias_retraso_pago",
    "impago_flag",
    "importe_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",
    "media_importe_3m",
    "media_retraso_3m",
    "impagos_3m",
    "cambio_consumo",
    "subida_factura",
    "variacion_consumo_pct",
    "stress_calidad_lag",
    "incidencia_masiva_lag"
]

# ============================================================
# X / y
# ============================================================

X = df_model[variables]
y = df_model["churn"]

# ============================================================
# TRAIN TEST
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================================
# SMOTE SOLO EN TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.45,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("="*60)
print("DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE")
print("="*60)

print(y_train_smote.value_counts())

# ============================================================
# MODELO
# ============================================================

modelo = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

modelo.fit(X_train_smote, y_train_smote)

# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]

# ============================================================
# THRESHOLD AJUSTADO
# ============================================================

threshold = 0.65

y_pred = (y_prob >= threshold).astype(int)



DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE
churn
0    13669
1     6151
Name: count, dtype: int64


In [53]:
# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("RESULTADOS MODELO")
print("="*60)

print("\nAccuracy:")
print(round(accuracy * 100, 2), "%")

print("\nPrecision:")
print(round(precision_score(y_test, y_pred, zero_division=0), 4))

print("\nRecall:")
print(round(recall_score(y_test, y_pred, zero_division=0), 4))

print("\nF1-score:")
print(round(f1_score(y_test, y_pred, zero_division=0), 4))

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob), 4))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


RESULTADOS MODELO

Accuracy:
93.49 %

Precision:
0.0

Recall:
0.0

F1-score:
0.0

ROC-AUC:
0.5445

Matriz de confusión:
[[3417    1]
 [ 237    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      3418
           1       0.00      0.00      0.00       237

    accuracy                           0.93      3655
   macro avg       0.47      0.50      0.48      3655
weighted avg       0.87      0.93      0.90      3655

